# **Notebook 02 - Data Cleaning**

**Output:**
- `data/interim/transactions_clean.parquet` - sales transactions only, all customers (for product/revenue analysis)
- `data/interim/transactions_customer_level.parquet` - sales transactions with a valid `CustomerID` only (for customer modeling: RFM, CLV, churn)
- `data/interim/cancellations.parquet` - separated cancellation records (useful as a churn signal feature later)

**Cleaning Rule** (derived from Notebook 01 findings):
1. Separate cancellation (Invoice starts with `C`) into their own table
2. Drop non-product StockCode (POST, BANK CHARGES, M, ect.)
3. Drop zero/negative prices and quantities (for the sales table)
4. Investigate and decide on extreme outliers (Quantity > 10,000)
5. Strip whitespace and normalize `Description`
6. Create derived columns: `Revenue`, `Year`, `Month`, `DayOfWeek`, `Hour`
7. Save as parquet (faster reload, preserves dtypes)

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:,.2f}'.format)

RAW_PATH = Path('../data/raw/online_retail_II.csv')
INTERIM_DIR = Path('../data/interim/')
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load raw data

In [2]:
df = pd.read_csv(
    RAW_PATH,
    dtype={'Invoice': str, 'StockCode': str, 'Customer ID': str},
    parse_dates=['InvoiceDate']
)
df = df.rename(columns={'Customer ID': 'CustomerID'})

print(f'Raw shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns')

# Track how many rows we drop at each step
audit = [('raw_data', len(df))]

Raw shape: 1,067,371 rows x 8 columns


## 2. Separate cancellations

Cancellations (Invoice starts with `C`) are real-business events - we don't throw them away. We pull them into their own table so we can:
- Compute return-rate per customer as a feature later.
- Optionally net them against sales for revenue calculations.

But they shouldn't be mixed into the sales table because their negative quantities will skew aggregations.

In [3]:
is_cancellation = df['Invoice'].str.startswith('C')

cancellations = df[is_cancellation].copy()
df = df[~is_cancellation].copy()

print(f'Cancellations separated: {len(cancellations):,} rows')
print(f'Sales remaining: {len(df):,} rows')
audit.append(('after_separating_cancellations', len(df)))

Cancellations separated: 19,494 rows
Sales remaining: 1,047,877 rows


## 3. Drop non-product StockCode

From Notebook 01, we identified admin/fee codes that aren't real products. We define an explicit list rather than using a heuristic - explicit is safer and reviewable

**Common non-product codes in this dateset:**
- `POST`, `DOT` - postage/shipping charges
- `M` - manual adjustments
- `BANK CHARGES`, `AMAZONFEE` - Fees
- `D` - discount 
- `PADS` - pads to match all sales
- `CRUK` - charity donation
- `B` - adjust bad debt
- `TEST001`, `TEST002`, `gift_0001` - test/gift card placeholders

In [4]:
for code in ['S', 'ADJUST']:
    rows = df[df['StockCode'] == code]
    print(f"\n=== {code} ({len(rows)} rows) ===")
    print(f"Description: {rows['Description'].value_counts().head(5).to_dict()}")
    print(f"Price range: {rows['Price'].min()} to {rows['Price'].max()}")
    print(f"Quantity range: {rows['Quantity'].min()} to {rows['Quantity'].max()}")


=== S (3 rows) ===
Description: {'SAMPLES': 3}
Price range: 30.0 to 73.8
Quantity range: 1 to 1

=== ADJUST (36 rows) ===
Description: {'Adjustment by john on 26/01/2010 16': 20, 'Adjustment by john on 26/01/2010 17': 16}
Price range: 4.57 to 5117.03
Quantity range: 1 to 1


In [5]:
NON_PRODUCT_CODES = ['POST', 'DOT', 'M', 'BANK CHARGES', 'D', 'C2', 'PADS', 'AMAZONFEE', 'CRUK', 'B', 'TEST001', 'TEST002', 'S', 'ADJUST']

before = len(df)
df = df[~df['StockCode'].isin(NON_PRODUCT_CODES)].copy()

# Also drop any StockCode starting with 'gift_' (gift card placeholders)
df = df[~df['StockCode'].str.startswith('gift_')].copy()

dropped = before - len(df)
print(f'Dropped {dropped:,} rows of non-product StockCodes')
print(f'Remaining: {len(df):,} rows')
audit.append(('after_dropping_non_products', len(df)))

Dropped 4,713 rows of non-product StockCodes
Remaining: 1,043,164 rows
